In [ ]:
# ============================================================
# EXPERIMENT 16 - OUTLIER TREATMENT
# CELL 0: INITIALIZATION & DATA VALIDATION
# ============================================================

import sys
import json
from pathlib import Path

ROOT_DIR = Path.cwd().resolve().parents[2]

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import src.imports as lib

from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    hamming_loss
)

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

RANDOM_SEED = 42
N_SPLITS = 5

lib.np.random.seed(RANDOM_SEED)

id_col = "ma_hoc_sinh"

choice_cols = [
    "Vật lí",
    "Hóa học",
    "Sinh học",
    "Địa lí",
    "Giáo dục kinh tế và pháp luật (GDKT&PL)",
    "Tin học",
    "Công nghệ",
    "Âm nhạc",
    "Mĩ thuật"
]

MODEL_NAMES = [
    "Random Forest",
    "XGBoost",
    "CatBoost"
]

TREATMENT_NAMES = [
    "raw",
    "iqr_capping",
    "quantile_capping"
]

IQR_FACTOR = 1.5

QUANTILE_LOWER = 0.01
QUANTILE_UPPER = 0.99

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

split_dir = ROOT_DIR / "data/data-splits"

fold_dir = split_dir / "cv_folds_v1"

output_dir = ROOT_DIR / (
    "experiments/v2_optimization/"
    "05_outlier_treatment/results"
)

table_dir = output_dir / "tables"
figure_dir = output_dir / "figures"
config_dir = output_dir / "configs"

for directory in [
    table_dir,
    figure_dir,
    config_dir
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

# ------------------------------------------------------------
# LOAD DEVELOPMENT
# ------------------------------------------------------------

df_dev = lib.pd.read_csv(
    split_dir / "development.csv",
    encoding="utf-8-sig",
    dtype={id_col: str}
)

df_dev[id_col] = df_dev[id_col].str.strip()

assert len(df_dev) == 9000, (
    f"Development có {len(df_dev)} học sinh, "
    "không phải 9000."
)

assert df_dev[id_col].notna().all()
assert df_dev[id_col].ne("").all()
assert df_dev[id_col].is_unique

# ------------------------------------------------------------
# ORIGINAL FEATURES
# ------------------------------------------------------------

score_cols = [
    col for col in df_dev.columns
    if col.startswith((
        "L6_", "L7_", "L8_", "L9_"
    ))
    and col.endswith((
        "_hki", "_hkii", "_ca_nam"
    ))
]

X = df_dev[score_cols].copy()
y = df_dev[choice_cols].copy()

assert X.shape == (9000, 96)
assert y.shape == (9000, 9)

assert X.notna().all().all()
assert lib.np.isfinite(
    X.to_numpy(dtype=float)
).all()

assert y.isin([0, 1]).all().all()
assert y.sum(axis=1).eq(4).all()

# ------------------------------------------------------------
# LOAD SAVED CROSS-VALIDATION
# ------------------------------------------------------------

fold_data = []
fold_records = []

all_dev_ids = set(df_dev[id_col])

# Map ID -> vị trí dòng thực tế
id_to_position = {
    student_id: position
    for position, student_id
    in enumerate(df_dev[id_col])
}

for fold in range(1, N_SPLITS + 1):

    train_path = (
        fold_dir / f"fold_{fold}_train_ids.csv"
    )

    val_path = (
        fold_dir / f"fold_{fold}_validation_ids.csv"
    )

    train_ids = lib.pd.read_csv(
        train_path,
        encoding="utf-8-sig",
        dtype={id_col: str}
    )[id_col].str.strip()

    val_ids = lib.pd.read_csv(
        val_path,
        encoding="utf-8-sig",
        dtype={id_col: str}
    )[id_col].str.strip()

    # 1. Không có ID trống hoặc trùng
    assert train_ids.notna().all()
    assert val_ids.notna().all()

    assert train_ids.is_unique, (
        f"Fold {fold}: Trùng ID Training."
    )

    assert val_ids.is_unique, (
        f"Fold {fold}: Trùng ID Validation."
    )

    train_set = set(train_ids)
    val_set = set(val_ids)

    # 2. Không giao nhau
    assert train_set.isdisjoint(val_set), (
        f"Fold {fold}: Train và Validation giao nhau."
    )

    # 3. Không chứa ID ngoài Development
    assert train_set.issubset(all_dev_ids)
    assert val_set.issubset(all_dev_ids)

    # 4. Bao phủ toàn bộ Development
    assert train_set | val_set == all_dev_ids, (
        f"Fold {fold}: Không bao phủ toàn bộ Development."
    )

    # 5. Tạo vị trí dòng theo thứ tự trong Development
    train_idx = lib.np.array(
        sorted(
            id_to_position[sid]
            for sid in train_set
        ),
        dtype=int
    )

    val_idx = lib.np.array(
        sorted(
            id_to_position[sid]
            for sid in val_set
        ),
        dtype=int
    )

    assert len(train_idx) == len(train_ids)
    assert len(val_idx) == len(val_ids)

    assert (
        len(train_idx) + len(val_idx)
        == len(df_dev)
    )

    fold_data.append({
        "Fold": fold,
        "train_idx": train_idx,
        "val_idx": val_idx
    })

    fold_records.append({
        "Fold": fold,
        "Train": len(train_idx),
        "Validation": len(val_idx),
        "Total": len(train_idx) + len(val_idx),
        "Overlap": len(train_set & val_set),
        "Status": "PASS"
    })

fold_report = lib.pd.DataFrame(fold_records)

# Mỗi học sinh phải thuộc Validation đúng 1 lần
all_validation_positions = lib.np.concatenate([
    item["val_idx"]
    for item in fold_data
])

assert len(all_validation_positions) == len(df_dev)

assert lib.np.array_equal(
    lib.np.sort(all_validation_positions),
    lib.np.arange(len(df_dev))
), "Các Validation fold không tạo thành phân hoạch 5-fold."

print("=" * 65)
print("EXPERIMENT 16 - INITIALIZATION")
print("=" * 65)

print("Development:", df_dev.shape)
print("X:", X.shape)
print("y:", y.shape)

print("\nCROSS-VALIDATION:")
display(fold_report)

print("\nPASS: 5 Fold hợp lệ.")
print("PASS: Không thay đổi CV V1.")
print("PASS: Independent Test chưa được sử dụng.")

In [ ]:
# ============================================================
# CELL 1: RAW OUTLIER ANALYSIS
# ============================================================

raw_records = []
raw_feature_records = []

for fold_info in fold_data:

    fold = fold_info["Fold"]

    X_train = X.iloc[
        fold_info["train_idx"]
    ].copy()

    q1 = X_train.quantile(0.25)
    q3 = X_train.quantile(0.75)

    iqr = q3 - q1

    lower = q1 - IQR_FACTOR * iqr
    upper = q3 + IQR_FACTOR * iqr

    mask = (
        X_train.lt(lower)
        | X_train.gt(upper)
    )

    n_outliers = int(
        mask.to_numpy().sum()
    )

    n_students = int(
        mask.any(axis=1).sum()
    )

    raw_records.append({
        "Fold": fold,
        "Train_Students": len(X_train),
        "Total_Values": X_train.size,
        "Outlier_Values": n_outliers,
        "Outlier_Rate_%": (
            n_outliers / X_train.size * 100
        ),
        "Affected_Students": n_students,
        "Affected_Student_Rate_%": (
            n_students / len(X_train) * 100
        )
    })

    for feature in score_cols:

        raw_feature_records.append({
            "Fold": fold,
            "Feature": feature,
            "Outlier_Count": int(
                mask[feature].sum()
            ),
            "Outlier_Rate_%": (
                mask[feature].mean() * 100
            )
        })

raw_report = lib.pd.DataFrame(raw_records)

raw_feature_report = lib.pd.DataFrame(
    raw_feature_records
)

print("=" * 65)
print("RAW OUTLIER DISTRIBUTION")
print("=" * 65)

display(raw_report.round(4))

print("\nTRUNG BÌNH 5 FOLD")

display(
    raw_report[
        [
            "Outlier_Rate_%",
            "Affected_Student_Rate_%"
        ]
    ].mean().to_frame("Mean")
)

print("\nTOP 15 ĐẶC TRƯNG CÓ NHIỀU OUTLIER")

display(
    raw_feature_report
    .groupby("Feature", as_index=False)[
        "Outlier_Rate_%"
    ]
    .mean()
    .sort_values(
        "Outlier_Rate_%",
        ascending=False
    )
    .head(15)
    .round(4)
)

print("\nPASS: Đã thống kê Outlier.")

In [ ]:
# ============================================================
# CELL 2: IQR CAPPING
# ============================================================

def fit_iqr_thresholds(
    X_train,
    factor=1.5
):
    """
    Chỉ sử dụng Training để xác định ngưỡng.
    """

    q1 = X_train.quantile(0.25)
    q3 = X_train.quantile(0.75)

    iqr = q3 - q1

    lower = q1 - factor * iqr
    upper = q3 + factor * iqr

    return lib.pd.DataFrame({
        "Feature": X_train.columns,
        "Lower": lower.to_numpy(dtype=float),
        "Upper": upper.to_numpy(dtype=float)
    })


def apply_capping(X_data, thresholds):
    """
    Áp dụng ngưỡng đã được fit từ Training.
    Không tính lại ngưỡng từ Validation.
    """

    if thresholds is None:
        return X_data.copy()

    bounds = thresholds.set_index("Feature")

    lower = bounds["Lower"].reindex(
        X_data.columns
    )

    upper = bounds["Upper"].reindex(
        X_data.columns
    )

    assert lower.notna().all()
    assert upper.notna().all()
    assert (lower <= upper).all()

    return X_data.clip(
        lower=lower,
        upper=upper,
        axis=1
    )


# Kiểm tra trên Fold 1
fold_info = fold_data[0]

X_train = X.iloc[
    fold_info["train_idx"]
].copy()

X_val = X.iloc[
    fold_info["val_idx"]
].copy()

iqr_thresholds = fit_iqr_thresholds(
    X_train,
    factor=IQR_FACTOR
)

X_train_iqr = apply_capping(
    X_train,
    iqr_thresholds
)

X_val_iqr = apply_capping(
    X_val,
    iqr_thresholds
)

assert X_train_iqr.shape == X_train.shape
assert X_val_iqr.shape == X_val.shape

assert X_train_iqr.index.equals(X_train.index)
assert X_val_iqr.index.equals(X_val.index)

print("IQR CAPPING - FOLD 1")

display(iqr_thresholds.head(10))

print("\nTraining:", X_train_iqr.shape)
print("Validation:", X_val_iqr.shape)

print("\nPASS: IQR Capping hợp lệ.")

In [ ]:
# ============================================================
# CELL 3: QUANTILE CAPPING
# ============================================================

def fit_quantile_thresholds(
    X_train,
    lower_q=0.01,
    upper_q=0.99
):
    """
    Ngưỡng phân vị được fit chỉ trên Training.
    """

    assert 0 <= lower_q < upper_q <= 1

    lower = X_train.quantile(lower_q)
    upper = X_train.quantile(upper_q)

    return lib.pd.DataFrame({
        "Feature": X_train.columns,
        "Lower": lower.to_numpy(dtype=float),
        "Upper": upper.to_numpy(dtype=float)
    })


quantile_thresholds = fit_quantile_thresholds(
    X_train,
    lower_q=QUANTILE_LOWER,
    upper_q=QUANTILE_UPPER
)

X_train_quantile = apply_capping(
    X_train,
    quantile_thresholds
)

X_val_quantile = apply_capping(
    X_val,
    quantile_thresholds
)

assert X_train_quantile.shape == X_train.shape
assert X_val_quantile.shape == X_val.shape

assert X_train_quantile.index.equals(X_train.index)
assert X_val_quantile.index.equals(X_val.index)

print("QUANTILE CAPPING - FOLD 1")

display(quantile_thresholds.head(10))

print("\nTraining:", X_train_quantile.shape)
print("Validation:", X_val_quantile.shape)

print("\nPASS: Quantile Capping hợp lệ.")

In [ ]:
# ============================================================
# CELL 4: APPLY TREATMENT TO ALL 5 FOLDS
# ============================================================

def fit_thresholds(X_train, treatment):

    if treatment == "raw":
        return None

    if treatment == "iqr_capping":
        return fit_iqr_thresholds(
            X_train,
            factor=IQR_FACTOR
        )

    if treatment == "quantile_capping":
        return fit_quantile_thresholds(
            X_train,
            lower_q=QUANTILE_LOWER,
            upper_q=QUANTILE_UPPER
        )

    raise ValueError(
        f"Unknown treatment: {treatment}"
    )


def transform_features(X_data, thresholds):

    if thresholds is None:
        return X_data.copy()

    return apply_capping(
        X_data,
        thresholds
    )


def calculate_adjustment_statistics(
    X_before,
    X_after,
    fold,
    treatment,
    dataset
):

    before = X_before.to_numpy(dtype=float)
    after = X_after.to_numpy(dtype=float)

    changed = before != after

    absolute_change = lib.np.abs(after - before)

    n_changed = int(changed.sum())

    n_students_affected = int(
        changed.any(axis=1).sum()
    )

    return {
        "Fold": fold,
        "Treatment": treatment,
        "Dataset": dataset,
        "N_Students": len(X_before),
        "Total_Values": before.size,
        "Adjusted_Values": n_changed,
        "Adjusted_Rate_%": (
            100 * n_changed / before.size
        ),
        "Affected_Students": n_students_affected,
        "Affected_Student_Rate_%": (
            100 * n_students_affected
            / len(X_before)
        ),
        "Mean_Absolute_Change": float(
            absolute_change.mean()
        ),
        "Max_Absolute_Change": float(
            absolute_change.max()
        )
    }


processed_folds = {}

adjustment_records = []
feature_adjustment_records = []
threshold_records = []

for fold_info in fold_data:

    fold = fold_info["Fold"]

    train_idx = fold_info["train_idx"]
    val_idx = fold_info["val_idx"]

    X_train_raw = X.iloc[train_idx].copy()
    X_val_raw = X.iloc[val_idx].copy()

    for treatment in TREATMENT_NAMES:

        # FIT trên Training
        thresholds = fit_thresholds(
            X_train_raw,
            treatment
        )

        # TRANSFORM Train và Validation
        X_train_processed = transform_features(
            X_train_raw,
            thresholds
        )

        X_val_processed = transform_features(
            X_val_raw,
            thresholds
        )

        processed_folds[(fold, treatment)] = {
            "X_train": X_train_processed,
            "X_val": X_val_processed
        }

        # Lưu ngưỡng
        if thresholds is not None:

            temp = thresholds.copy()

            temp["Fold"] = fold
            temp["Treatment"] = treatment

            threshold_records.append(temp)

        # Thống kê Train / Validation riêng
        for dataset, before, after in [
            ("Train", X_train_raw, X_train_processed),
            ("Validation", X_val_raw, X_val_processed)
        ]:

            adjustment_records.append(
                calculate_adjustment_statistics(
                    before,
                    after,
                    fold,
                    treatment,
                    dataset
                )
            )

            changed_mask = (
                before.to_numpy(dtype=float)
                != after.to_numpy(dtype=float)
            )

            for j, feature in enumerate(score_cols):

                feature_adjustment_records.append({
                    "Fold": fold,
                    "Treatment": treatment,
                    "Dataset": dataset,
                    "Feature": feature,
                    "Adjusted_Values": int(
                        changed_mask[:, j].sum()
                    ),
                    "Adjusted_Rate_%": float(
                        changed_mask[:, j].mean() * 100
                    )
                })


adjustment_df = lib.pd.DataFrame(
    adjustment_records
)

feature_adjustment_df = lib.pd.DataFrame(
    feature_adjustment_records
)

threshold_df = lib.pd.concat(
    threshold_records,
    ignore_index=True
)

adjustment_summary = (
    adjustment_df
    .groupby(
        ["Treatment", "Dataset"],
        as_index=False
    )
    .agg(
        Mean_Adjusted_Rate=(
            "Adjusted_Rate_%", "mean"
        ),
        Mean_Affected_Student_Rate=(
            "Affected_Student_Rate_%", "mean"
        ),
        Mean_Absolute_Change=(
            "Mean_Absolute_Change", "mean"
        ),
        Mean_Max_Absolute_Change=(
            "Max_Absolute_Change", "mean"
        )
    )
)

print("=" * 65)
print("OUTLIER ADJUSTMENT SUMMARY")
print("=" * 65)

display(adjustment_summary.round(4))

print("\nTHỐNG KÊ CHI TIẾT")
display(adjustment_df.round(4))

print("\nTOP 15 FEATURES BỊ ĐIỀU CHỈNH NHIỀU NHẤT")

top_adjusted_features = (
    feature_adjustment_df[
        (feature_adjustment_df["Dataset"] == "Train")
        & (feature_adjustment_df["Treatment"] != "raw")
    ]
    .groupby(
        ["Treatment", "Feature"],
        as_index=False
    )["Adjusted_Rate_%"]
    .mean()
    .sort_values(
        "Adjusted_Rate_%",
        ascending=False
    )
    .head(15)
)

display(top_adjusted_features.round(4))

print("\nPASS: Đã xử lý 5 fold.")

In [ ]:
# ============================================================
# CELL 5: DATA INTEGRITY VALIDATION
# ============================================================

integrity_records = []

for fold_info in fold_data:

    fold = fold_info["Fold"]

    train_idx = fold_info["train_idx"]
    val_idx = fold_info["val_idx"]

    X_train_original = X.iloc[train_idx]
    X_val_original = X.iloc[val_idx]

    y_train_original = y.iloc[train_idx]
    y_val_original = y.iloc[val_idx]

    for treatment in TREATMENT_NAMES:

        processed = processed_folds[
            (fold, treatment)
        ]

        X_train_processed = processed["X_train"]
        X_val_processed = processed["X_val"]

        # Không thay đổi số học sinh
        assert (
            X_train_processed.shape
            == X_train_original.shape
        )

        assert (
            X_val_processed.shape
            == X_val_original.shape
        )

        # Không thay đổi thứ tự dòng
        assert X_train_processed.index.equals(
            X_train_original.index
        )

        assert X_val_processed.index.equals(
            X_val_original.index
        )

        # Không thay đổi cột
        assert X_train_processed.columns.equals(
            X_train_original.columns
        )

        assert X_val_processed.columns.equals(
            X_val_original.columns
        )

        # Không tạo missing values
        assert lib.np.isfinite(
            X_train_processed.to_numpy(dtype=float)
        ).all()

        assert lib.np.isfinite(
            X_val_processed.to_numpy(dtype=float)
        ).all()

        # Nhãn gốc không bị thay đổi
        assert y_train_original.sum(axis=1).eq(
            4
        ).all()

        assert y_val_original.sum(axis=1).eq(
            4
        ).all()

        if treatment == "raw":

            lib.pd.testing.assert_frame_equal(
                X_train_processed,
                X_train_original
            )

            lib.pd.testing.assert_frame_equal(
                X_val_processed,
                X_val_original
            )

        integrity_records.append({
            "Fold": fold,
            "Treatment": treatment,
            "Train_Before": len(X_train_original),
            "Train_After": len(X_train_processed),
            "Validation_Before": len(X_val_original),
            "Validation_After": len(X_val_processed),
            "Students_Removed": 0,
            "Status": "PASS"
        })


integrity_df = lib.pd.DataFrame(
    integrity_records
)

display(integrity_df)

assert len(integrity_df) == (
    N_SPLITS * len(TREATMENT_NAMES)
)

assert integrity_df["Status"].eq("PASS").all()

print("\nPASS: Không xóa hoặc thêm học sinh.")
print("PASS: Không thay đổi nhãn mục tiêu.")
print("PASS: Không thay đổi 5 fold.")

In [ ]:
# ============================================================
# CELL 6: MODEL TRAINING - 5 FOLD CV
# ============================================================

config_paths = {
    "Random Forest": ROOT_DIR / (
        "experiments/v1_baseline/"
        "random_forest/results/rf_config.json"
    ),
    "XGBoost": ROOT_DIR / (
        "experiments/v1_baseline/"
        "xgboost/results/xgb_config.json"
    ),
    "CatBoost": ROOT_DIR / (
        "experiments/v1_baseline/"
        "catboost/results/cat_config.json"
    )
}

v1_params = {}

for model_name, path in config_paths.items():

    with open(path, "r", encoding="utf-8") as f:
        config = json.load(f)

    v1_params[model_name] = config["hyperparameters"]


def create_model(model_name):

    params = v1_params[model_name].copy()

    # Giữ cấu hình không gán trọng số như V1
    if model_name == "Random Forest":

        params["class_weight"] = None

        base_model = RandomForestClassifier(
            **params
        )

    elif model_name == "XGBoost":

        params["scale_pos_weight"] = 1.0

        base_model = XGBClassifier(
            **params
        )

    elif model_name == "CatBoost":

        params.pop("class_weights", None)
        params.pop("auto_class_weights", None)

        base_model = CatBoostClassifier(
            **params
        )

    else:
        raise ValueError(
            f"Unknown model: {model_name}"
        )

    return MultiOutputClassifier(
        estimator=base_model,
        n_jobs=1
    )


def predict_positive_proba(model, X_val):

    probabilities = []

    for estimator in model.estimators_:

        classes = list(estimator.classes_)

        assert 1 in classes, (
            "Một bộ phân loại không có lớp dương."
        )

        positive_index = classes.index(1)

        prob = estimator.predict_proba(
            X_val
        )[:, positive_index]

        probabilities.append(prob)

    return lib.np.column_stack(
        probabilities
    )


def predict_top4(probabilities):

    y_pred = lib.np.zeros_like(
        probabilities,
        dtype=int
    )

    top4_indices = lib.np.argsort(
        -probabilities,
        axis=1,
        kind="stable"
    )[:, :4]

    rows = lib.np.arange(
        probabilities.shape[0]
    )[:, None]

    y_pred[rows, top4_indices] = 1

    return y_pred


experiment_results = []

for model_name in MODEL_NAMES:

    for treatment in TREATMENT_NAMES:

        print("=" * 65)
        print(f"{model_name} | {treatment}")
        print("=" * 65)

        for fold_info in fold_data:

            fold = fold_info["Fold"]

            train_idx = fold_info["train_idx"]
            val_idx = fold_info["val_idx"]

            processed = processed_folds[
                (fold, treatment)
            ]

            X_train = processed["X_train"]
            X_val = processed["X_val"]

            y_train = y.iloc[train_idx]
            y_val = y.iloc[val_idx]

            # Mỗi fold dùng một mô hình mới
            model = create_model(model_name)

            model.fit(
                X_train,
                y_train
            )

            y_proba = predict_positive_proba(
                model,
                X_val
            )

            y_pred = predict_top4(
                y_proba
            )

            y_true = y_val.to_numpy(
                dtype=int
            )

            assert y_proba.shape == (
                len(X_val), 9
            )

            assert lib.np.isfinite(
                y_proba
            ).all()

            assert (
                y_pred.sum(axis=1) == 4
            ).all()

            exact_match = accuracy_score(
                y_true,
                y_pred
            )

            macro_f1 = f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            )

            experiment_results.append({
                "Model": model_name,
                "Treatment": treatment,
                "Fold": fold,
                "val_idx": val_idx,
                "y_true": y_true,
                "y_pred": y_pred,
                "y_proba": y_proba
            })

            print(
                f"Fold {fold} | "
                f"Train={len(X_train)} | "
                f"Validation={len(X_val)} | "
                f"Exact={exact_match:.4f} | "
                f"Macro-F1={macro_f1:.4f}"
            )

assert len(experiment_results) == (
    len(MODEL_NAMES)
    * len(TREATMENT_NAMES)
    * N_SPLITS
)

print("\nPASS: Hoàn thành huấn luyện 3 mô hình.")
print("Tổng lượt đánh giá:", len(experiment_results))

In [ ]:
# ============================================================
# CELL 7: EVALUATION & COMPARISON
# ============================================================

metric_records = []
label_records = []
prediction_frames = []

for result in experiment_results:

    info = {
        "Model": result["Model"],
        "Treatment": result["Treatment"],
        "Fold": result["Fold"]
    }

    y_true = result["y_true"]
    y_pred = result["y_pred"]
    y_proba = result["y_proba"]

    metric_records.append({
        **info,
        "Exact_Match_Accuracy": accuracy_score(
            y_true,
            y_pred
        ),
        "Macro_F1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "Micro_F1": f1_score(
            y_true,
            y_pred,
            average="micro",
            zero_division=0
        ),
        "Hamming_Loss": hamming_loss(
            y_true,
            y_pred
        )
    })

    label_scores = f1_score(
        y_true,
        y_pred,
        average=None,
        zero_division=0
    )

    for label, score in zip(
        choice_cols,
        label_scores
    ):

        label_records.append({
            **info,
            "Nhãn": label,
            "F1": float(score)
        })

    pred_df = lib.pd.DataFrame({
        id_col: df_dev.iloc[
            result["val_idx"]
        ][id_col].to_numpy(),
        **info
    })

    for j in range(9):

        pred_df[f"true_{j+1}"] = y_true[:, j]
        pred_df[f"pred_{j+1}"] = y_pred[:, j]
        pred_df[f"proba_{j+1}"] = y_proba[:, j]

    prediction_frames.append(pred_df)


ot_metrics = lib.pd.DataFrame(
    metric_records
)

ot_label_f1 = lib.pd.DataFrame(
    label_records
)

ot_predictions = lib.pd.concat(
    prediction_frames,
    ignore_index=True
)

# ------------------------------------------------------------
# SUMMARY: MEAN & STD
# ------------------------------------------------------------

ot_summary = (
    ot_metrics
    .groupby(
        ["Model", "Treatment"]
    )
    .agg(
        Exact_Match_Mean=(
            "Exact_Match_Accuracy", "mean"
        ),
        Exact_Match_Std=(
            "Exact_Match_Accuracy", "std"
        ),
        Macro_F1_Mean=(
            "Macro_F1", "mean"
        ),
        Macro_F1_Std=(
            "Macro_F1", "std"
        ),
        Micro_F1_Mean=(
            "Micro_F1", "mean"
        ),
        Hamming_Loss_Mean=(
            "Hamming_Loss", "mean"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# COMPARE WITH RAW
# ------------------------------------------------------------

raw_summary = (
    ot_summary[
        ot_summary["Treatment"] == "raw"
    ][
        [
            "Model",
            "Exact_Match_Mean",
            "Macro_F1_Mean",
            "Hamming_Loss_Mean"
        ]
    ]
    .rename(columns={
        "Exact_Match_Mean": "Raw_Exact",
        "Macro_F1_Mean": "Raw_Macro_F1",
        "Hamming_Loss_Mean": "Raw_Hamming"
    })
)

ot_comparison = ot_summary.merge(
    raw_summary,
    on="Model",
    how="left",
    validate="many_to_one"
)

ot_comparison["Delta_Exact_vs_Raw"] = (
    ot_comparison["Exact_Match_Mean"]
    - ot_comparison["Raw_Exact"]
)

ot_comparison["Delta_Macro_vs_Raw"] = (
    ot_comparison["Macro_F1_Mean"]
    - ot_comparison["Raw_Macro_F1"]
)

ot_comparison["Delta_Hamming_vs_Raw"] = (
    ot_comparison["Hamming_Loss_Mean"]
    - ot_comparison["Raw_Hamming"]
)

# ------------------------------------------------------------
# PAIRED-FOLD COMPARISON
# ------------------------------------------------------------

raw_fold = (
    ot_metrics[
        ot_metrics["Treatment"] == "raw"
    ][
        [
            "Model",
            "Fold",
            "Exact_Match_Accuracy",
            "Macro_F1"
        ]
    ]
    .rename(columns={
        "Exact_Match_Accuracy": "Raw_Fold_Exact",
        "Macro_F1": "Raw_Fold_Macro"
    })
)

paired_comparison = ot_metrics.merge(
    raw_fold,
    on=["Model", "Fold"],
    how="left",
    validate="many_to_one"
)

paired_comparison["Delta_Exact"] = (
    paired_comparison["Exact_Match_Accuracy"]
    - paired_comparison["Raw_Fold_Exact"]
)

paired_comparison["Delta_Macro"] = (
    paired_comparison["Macro_F1"]
    - paired_comparison["Raw_Fold_Macro"]
)

print("=" * 65)
print("OUTLIER TREATMENT - COMPARISON")
print("=" * 65)

display(
    ot_comparison.sort_values(
        ["Model", "Exact_Match_Mean"],
        ascending=[True, False]
    ).round(4)
)

print("\nPAIRED COMPARISON BY FOLD")

display(
    paired_comparison[
        [
            "Model",
            "Treatment",
            "Fold",
            "Delta_Exact",
            "Delta_Macro"
        ]
    ].round(4)
)

# ------------------------------------------------------------
# PER-LABEL F1
# ------------------------------------------------------------

label_comparison = (
    ot_label_f1
    .groupby(
        ["Model", "Treatment", "Nhãn"],
        as_index=False
    )["F1"]
    .mean()
)

for model_name in MODEL_NAMES:

    print(f"\nF1 TỪNG NHÃN — {model_name}")

    display(
        label_comparison[
            label_comparison["Model"] == model_name
        ].pivot(
            index="Nhãn",
            columns="Treatment",
            values="F1"
        ).round(4)
    )

print("\nPASS: Đã đánh giá 3 phương pháp xử lý.")

In [ ]:
# ============================================================
# CELL 8: VERIFY AGAINST V1 BASELINE
# ============================================================

v1_files = {
    "Random Forest": ROOT_DIR / (
        "experiments/v1_baseline/"
        "random_forest/results/rf_metrics_5fold.csv"
    ),
    "XGBoost": ROOT_DIR / (
        "experiments/v1_baseline/"
        "xgboost/results/xgb_metrics_5fold.csv"
    ),
    "CatBoost": ROOT_DIR / (
        "experiments/v1_baseline/"
        "catboost/results/cat_metrics_5fold.csv"
    )
}

v1_records = []

for model_name, path in v1_files.items():

    v1_df = lib.pd.read_csv(
        path,
        encoding="utf-8-sig"
    )

    v1_records.append({
        "Model": model_name,
        "V1_Exact": v1_df[
            "Exact_Match_Accuracy"
        ].mean(),
        "V1_Macro_F1": v1_df[
            "Macro_F1"
        ].mean()
    })

v1_summary = lib.pd.DataFrame(v1_records)

ot_comparison = ot_comparison.merge(
    v1_summary,
    on="Model",
    how="left",
    validate="many_to_one"
)

ot_comparison["Delta_Exact_vs_V1"] = (
    ot_comparison["Exact_Match_Mean"]
    - ot_comparison["V1_Exact"]
)

ot_comparison["Delta_Macro_vs_V1"] = (
    ot_comparison["Macro_F1_Mean"]
    - ot_comparison["V1_Macro_F1"]
)

raw_check = ot_comparison[
    ot_comparison["Treatment"] == "raw"
].copy()

raw_check["Exact_Difference"] = (
    raw_check["Exact_Match_Mean"]
    - raw_check["V1_Exact"]
)

raw_check["Macro_Difference"] = (
    raw_check["Macro_F1_Mean"]
    - raw_check["V1_Macro_F1"]
)

print("=" * 65)
print("RAW BASELINE REPRODUCIBILITY CHECK")
print("=" * 65)

display(
    raw_check[
        [
            "Model",
            "Exact_Match_Mean",
            "V1_Exact",
            "Exact_Difference",
            "Macro_F1_Mean",
            "V1_Macro_F1",
            "Macro_Difference"
        ]
    ].round(6)
)

# Ngưỡng kiểm tra sai khác thực nghiệm
REPRODUCTION_TOLERANCE = 0.005

reproduction_pass = (
    raw_check["Exact_Difference"].abs().le(
        REPRODUCTION_TOLERANCE
    ).all()
    and
    raw_check["Macro_Difference"].abs().le(
        REPRODUCTION_TOLERANCE
    ).all()
)

if reproduction_pass:
    print("\nPASS: Raw gần tương đương V1.")
else:
    print(
        "\nWARNING: Raw khác V1 vượt ngưỡng 0.005. "
        "Cần kiểm tra trước khi kết luận."
    )

In [ ]:
# ============================================================
# CELL 9: VISUALIZATION & SAVE RESULTS
# ============================================================

# ------------------------------------------------------------
# PLOT 1: EXACT MATCH ACCURACY
# ------------------------------------------------------------

plot_exact = ot_summary.pivot(
    index="Treatment",
    columns="Model",
    values="Exact_Match_Mean"
) * 100

ax = plot_exact.plot(
    kind="bar",
    figsize=(11, 6)
)

ax.set_title(
    "Outlier Treatment - Exact-match Accuracy"
)
ax.set_xlabel("Treatment")
ax.set_ylabel("Exact-match Accuracy (%)")

lib.plt.xticks(rotation=0)
lib.plt.tight_layout()

fig = ax.get_figure()

fig.savefig(
    figure_dir / "outlier_exact_match.png",
    dpi=200,
    bbox_inches="tight"
)

lib.plt.show()
lib.plt.close(fig)

# ------------------------------------------------------------
# PLOT 2: MACRO F1
# ------------------------------------------------------------

plot_macro = ot_summary.pivot(
    index="Treatment",
    columns="Model",
    values="Macro_F1_Mean"
)

ax = plot_macro.plot(
    kind="bar",
    figsize=(11, 6)
)

ax.set_title(
    "Outlier Treatment - Macro F1"
)
ax.set_xlabel("Treatment")
ax.set_ylabel("Macro F1")

lib.plt.xticks(rotation=0)
lib.plt.tight_layout()

fig = ax.get_figure()

fig.savefig(
    figure_dir / "outlier_macro_f1.png",
    dpi=200,
    bbox_inches="tight"
)

lib.plt.show()
lib.plt.close(fig)

# ------------------------------------------------------------
# SAVE TABLES
# ------------------------------------------------------------

reports = {
    "cv_fold_validation.csv": fold_report,
    "raw_outlier_statistics.csv": raw_report,
    "raw_outlier_features.csv": raw_feature_report,
    "outlier_adjustment_5fold.csv": adjustment_df,
    "outlier_adjustment_summary.csv": adjustment_summary,
    "outlier_feature_adjustment.csv": feature_adjustment_df,
    "outlier_thresholds_5fold.csv": threshold_df,
    "outlier_data_integrity.csv": integrity_df,
    "outlier_metrics_5fold.csv": ot_metrics,
    "outlier_metrics_summary.csv": ot_summary,
    "outlier_comparison_vs_raw_v1.csv": ot_comparison,
    "outlier_paired_fold_comparison.csv": paired_comparison,
    "outlier_label_f1.csv": ot_label_f1,
    "outlier_label_f1_comparison.csv": label_comparison,
    "outlier_predictions_5fold.csv": ot_predictions
}

for filename, dataframe in reports.items():

    dataframe.to_csv(
        table_dir / filename,
        index=False,
        encoding="utf-8-sig"
    )

# ------------------------------------------------------------
# SAVE CONFIG
# ------------------------------------------------------------

experiment_config = {
    "experiment": "16 - Outlier Treatment",
    "random_seed": RANDOM_SEED,
    "n_splits": N_SPLITS,
    "cv_source": "data/data-splits/cv_folds_v1",
    "n_development": len(df_dev),
    "n_features": len(score_cols),
    "n_labels": len(choice_cols),
    "models": MODEL_NAMES,
    "treatments": {
        "raw": {
            "method": "None"
        },
        "iqr_capping": {
            "method": "IQR",
            "factor": IQR_FACTOR
        },
        "quantile_capping": {
            "method": "Quantile",
            "lower": QUANTILE_LOWER,
            "upper": QUANTILE_UPPER
        }
    },
    "fit_thresholds_on": "Training fold only",
    "validation_uses_training_thresholds": True,
    "students_removed": 0,
    "target_modified": False,
    "prediction_strategy": "Top-4",
    "class_weight_enabled": False,
    "oversampling_enabled": False,
    "feature_engineering_enabled": False,
    "v1_hyperparameters": v1_params,
    "raw_reproduction_pass": bool(reproduction_pass),
    "independent_test_used": False
}

with open(
    config_dir / "outlier_treatment_config.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        experiment_config,
        f,
        ensure_ascii=False,
        indent=4
    )

print("=" * 65)
print("EXPERIMENT 16 - COMPLETED")
print("=" * 65)

print("Báo cáo CSV:", len(reports))
print("Biểu đồ:", 2)
print("Số lượt huấn luyện:", len(experiment_results))
print("Raw tái lập V1:", reproduction_pass)
print("Số học sinh bị loại:", 0)
print("Independent Test sử dụng:", False)

print("\nThư mục:")
print(output_dir.resolve())

print("\nPASS: Đã lưu kết quả.")